# CS-Jail-UR: Stage C (judge gates) + Exp 2 judging, CPU only, resumable

Runs headless with **Save Version → Save & Run All (Commit)**. No GPU: the judge
is the OpenAI API and Exp 2 responses are judged from the cache. The logic is
`scripts/stagec_run.py` (tested in `tests/test_stagec.py`).

**Settings:** Accelerator **None**, Internet **On**; Add-ons → Secrets →
`OPENAI_API_KEY` (paid key) with **Attached** ticked.

**First run:** Add Input → your *private* dataset made from `stageC_upload.zip`.

**Continuing a run** (budget ran out, time limit reached, or new adjudications):
add the previous version's `stageC_outputs.zip` as an input as well (Add Input
→ Notebook Output → this notebook's previous version, or upload the zip as a
private dataset), then Save & Run All again. Finished work is never repeated
or paid for twice; the newest bundle wins.

Order: harm gate → benign gate (if uploaded) → Exp 2 main judging (only after a
harm PASS) → Exp 2 robustness judging (only after main is complete). Every step
stops at the time budget; the resume bundle `stageC_outputs.zip` is **always**
written. Download it from the version's **Output** tab. Keep this notebook
private: its outputs contain prompts, responses and judge rationales.

In [ ]:
# Settings for "Save & Run All".
RUBRIC_DEV_ONLY = True    # True: judge only the development set (rubric iteration, ~$2.5, ~35 min)
                          # False: harm gate -> benign gate -> Exp 2 judging (needs the
                          #        annotated validation-v2-merged folder in the upload)
TIME_BUDGET_H = 11.0      # Kaggle stops a run at 12 h; no step runs past this budget
JUDGE_ROBUSTNESS = True   # uses the time left after Exp 2 main; continues in the next run

import os
from IPython import get_ipython


def run(cmd):
    """Run a shell command with streamed output; return its exit code."""
    print("$", cmd, flush=True)
    get_ipython().system(cmd)
    return get_ipython().user_ns.get("_exit_code", 0)


def sh(cmd):
    """Like run(), but stop here if setup fails (nothing has been judged yet)."""
    code = run(cmd)
    if code:
        raise RuntimeError(f"setup step failed (exit {code}): {cmd}")

In [ ]:
# Repo + Python 3.12 environment (judge and test packages only; no GPU stack).
%cd /kaggle/working
if not os.path.isdir("NLP_Research"):
    sh("git clone --branch revision/final-692-dataset https://github.com/mshamoonbutt/NLP_Research.git")
%cd /kaggle/working/NLP_Research
sh("git pull --ff-only && git log --oneline -1")
sh("python -m pip install -q uv")
sh("python -m uv venv --allow-existing --python 3.12 /root/py312")
sh('python -m uv pip install --python /root/py312/bin/python -e ".[judge,dev]"')
os.environ["PATH"] = "/root/py312/bin:" + os.environ["PATH"]
os.environ["VIRTUAL_ENV"] = "/root/py312"
os.environ.pop("PYTHONPATH", None)
sh("which python && python --version")

In [ ]:
# The paid key from Kaggle Secrets (never typed into a cell).
from kaggle_secrets import UserSecretsClient
try:
    os.environ["OPENAI_API_KEY"] = UserSecretsClient().get_secret("OPENAI_API_KEY")
except Exception as e:
    raise RuntimeError("Attach the OPENAI_API_KEY secret (Add-ons -> Secrets) and re-run") from e
print("OPENAI_API_KEY loaded")

In [ ]:
# Gates + Exp 2 judging. Always ends by writing /kaggle/working/stageC_outputs.zip.
cmd = ("python scripts/stagec_run.py --input-root /kaggle/input "
       "--out-zip /kaggle/working/stageC_outputs.zip --time-budget-h " + str(TIME_BUDGET_H))
if not JUDGE_ROBUSTNESS:
    cmd += " --no-robustness"
if RUBRIC_DEV_ONLY:
    cmd += " --dev-sample-dir outputs/exp1/rubric-dev-01"
print("stagec_run exit:", run(cmd))

In [ ]:
# Summary of this run (full details are in the zip's STAGEC_STATUS.json).
import json
import zipfile
with zipfile.ZipFile("/kaggle/working/stageC_outputs.zip") as z:
    st = json.loads(z.read("STAGEC_STATUS.json"))
print("mode:", st.get("mode"), "| judge:", st.get("judge"))
print("harm gate PASS:", st["harm_gate_pass"])
for k, v in st["steps"].items():
    print(f"  {k}: {v}")
print("Exp 2 judgments by status:", st["judgments"])
print("errors:", st["errors"] or "none")
print("Step codes: 0 done/PASS, 1 FAIL, 2 INSUFFICIENT_EVIDENCE, 3 input error, "
      "4 incomplete (API/budget), 124 time budget reached. 4 and 124 resume next run.")